In [14]:
using CSV
using DataFrames
using Dates
using JSON
using XLSX
using StatsBase

In [15]:
function print_dict(d::Dict)
    for (k, v) in d
        println("$(k) => $(v)")
    end
end

print_dict (generic function with 1 method)

In [16]:
#= folder = "../instances_xlsx/A_MTN_3/"
file = "../instances_xlsx/A_MTN_3/224FL_10A_2.xlsx" =#
file = "AS_2024-01_useful.xlsx"
df_flights = DataFrame(XLSX.readtable(file, "Data"))

Row,YEAR,MONTH,DAY,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,DEPARTURE_TIME,AIR_TIME,ARRIVAL_TIME
,Any,Any,Any,Any,Any,Any,Any,Any,Any
1,2024,1,1,N526AS,ANC,SEA,27,175,281
2,2024,1,1,N607AS,ANC,SEA,137,173,389
3,2024,1,1,N585AS,ANC,SEA,162,174,428
4,2024,1,1,N317AS,FAI,SEA,201,177,455
5,2024,1,1,N587AS,JNU,SEA,310,116,514
6,2024,1,1,N284AK,PHX,PDX,356,138,453
7,2024,1,1,N320AS,SJC,SEA,364,106,507
8,2024,1,1,N277AK,SEA,FAT,375,113,503
9,2024,1,1,N588AS,ORD,PDX,377,225,534


In [21]:
tail_numbers = unique(df_flights.TAIL_NUMBER)
temp = sum(df_flights.AIR_TIME)
maximum(df_flights.AIR_TIME)
temp

660887

In [18]:
# air_time_by_tail_day est la somme des airtime de chaque tail pour chaque jour
air_time_by_tail_day = Dict{Tuple{String, Int}, Float64}()

# takeoff_by_tail_day est le nombre d'occurence de chaque tail number pour chaque jour
takeoff_by_tail_day = Dict{Tuple{String, Int}, Int}()
for row in eachrow(df_flights)
    tail = row.TAIL_NUMBER
    day = row.DAY
    airtime = row.AIR_TIME

    key = (tail, day)

    if haskey(air_time_by_tail_day, key)
        air_time_by_tail_day[key] += airtime
        takeoff_by_tail_day[key] += 1
    else
        air_time_by_tail_day[key] = airtime
        takeoff_by_tail_day[key] = 1
    end
end

result_df = DataFrame(
    TAIL_NUMBER = String[],
    DAY = Int[],
    TAKEOFF = Int[],
    FLYING_TIME = Float64[]
)

for ((tail, day), takeoff) in sort(collect(takeoff_by_tail_day); by=x->(x[1][1], x[1][2]))
    flying_time = air_time_by_tail_day[(tail, day)]
    push!(result_df, (tail, day, takeoff, flying_time))
end

result_df

Row,TAIL_NUMBER,DAY,TAKEOFF,FLYING_TIME
,String,Int64,Int64,Float64
1,N215AK,1,3,312.0
2,N215AK,2,3,789.0
3,N215AK,3,3,860.0
4,N215AK,4,3,658.0
5,N215AK,5,4,655.0
6,N215AK,6,2,622.0
7,N215AK,7,3,949.0
8,N215AK,8,4,702.0
9,N215AK,9,3,592.0


In [22]:
println("Average flying time per aircraft per day: ", round(sum(result_df.FLYING_TIME)/length(tail_numbers)/31, digits=2), " minutes")
println("Average takeoffs per aircraft per day: ", round(sum(result_df.TAKEOFF)/length(tail_numbers)/31, digits=2), " takeoffs")
println("Average flying time per takeoff: ", round(sum(result_df.FLYING_TIME)/sum(result_df.TAKEOFF), digits=2), " minutes")
#= println("Average flying time per aircraft per week: ", round(sum(result_df.FLYING_TIME)/length(tail_numbers)/4, digits=2), " minutes")
println("Average takeoffs per aircraft per week: ", round(sum(result_df.TAKEOFF)/length(tail_numbers)/4, digits=2), " minutes")
=#

Average flying time per aircraft per day: 495.79 minutes
Average takeoffs per aircraft per day: 2.88 takeoffs
Average flying time per takeoff: 172.42 minutes
